#### 데이터 로드

In [1]:
import torch
import torch.nn as nn
from spam_data import load_sms_spam
from textutils import tokenize_en, build_vocab, pad_and_tensor
from model import SpamRNN

VOCAB_SIZE = 2000
MAX_LEN = 50

train, val = load_sms_spam()

/home/pc22/llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 토큰화

In [2]:
train_toks = [tokenize_en(t) for t in train["sms"]]
val_toks = [tokenize_en(t) for t in val["sms"]]

In [3]:
word2idx, counter = build_vocab(
    train_toks,
    max_size=VOCAB_SIZE
)

X_train = pad_and_tensor(
    train_toks,
    word2idx,
    MAX_LEN
)

y_train = torch.tensor(
    train["label"],
    dtype=torch.float32
)

X_val = pad_and_tensor(
    val_toks,
    word2idx,
    MAX_LEN
)

y_val = torch.tensor(
    val["label"],
    dtype=torch.float32
)

# 배치 DataLoader
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

#### 모델

In [4]:
model = SpamRNN(VOCAB_SIZE)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

#### 학습

In [7]:
MAX_EPOCH = 10

for epoch in range(MAX_EPOCH):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        pred = model(xb)
        loss = criterion(pred, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"epoch {epoch + 1:02d} | "
        f"loss {avg_loss:.4f}"
    )

epoch 01 | loss 0.4676
epoch 02 | loss 0.3041
epoch 03 | loss 0.1887
epoch 04 | loss 0.1342
epoch 05 | loss 0.1078
epoch 06 | loss 0.0880
epoch 07 | loss 0.0765
epoch 08 | loss 0.0684
epoch 09 | loss 0.0608
epoch 10 | loss 0.0538


In [8]:
vtotal = 0.0
correct = 0
n = 0

# 검증
with torch.no_grad():
    model.eval()
    for xb, yb in val_loader:
        pred = model(xb)
        vtotal += criterion(pred, yb).item()
        correct += ((pred > 0.5).float() == yb).sum().item()
        n += len(yb)

val_loss = vtotal / len(val_loader)     # 손실
acc = correct / n                       # 정확도

        

In [9]:
print(val_loss, acc)

0.07372641651373771 0.97847533632287


In [10]:
def predict_spam(text):
    model.eval()

    tokens = tokenize_en(text)

    x = pad_and_tensor(
        [tokens],
        word2idx,
        MAX_LEN
    )

    with torch.no_grad():
        probability = model(x).item()

    label = "스팸" if probability >= 0.5 else "정상"

    return {
        "text": text,
        "prediction": label,
        "spam_probability": probability
    }

In [11]:
predict_spam(
    "Congratulations! You won a free prize. Call now!"
)

{'text': 'Congratulations! You won a free prize. Call now!',
 'prediction': '스팸',
 'spam_probability': 0.97765052318573}

In [12]:
predict_spam(
    "Are we still meeting for lunch tomorrow?"
)

{'text': 'Are we still meeting for lunch tomorrow?',
 'prediction': '정상',
 'spam_probability': 0.0037988328840583563}